# ⚛️ ECHO — Entangled Minds: The Quantathon
**Team:** _<your team name>_ **Member:** Inam
**Mandatory modules:** 01 · 03 · 05  **Stretch module:** 04 — The Erased Measurement

Each module below has its code, its result, and a one-line **"What it proves"**.
Run with **Runtime → Restart and run all**.

## 0 · Setup

In [ ]:
!pip install qiskit qiskit-aer pylatexenc --quiet
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np

sim = AerSimulator()
def run(qc, shots=1024):
    """Run a circuit many times and count the outcomes."""
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

print("✅ ready")

---
## Module 01 · The Gatekeeper *(mandatory)*
Gates allowed: H · X · measure.

### 🥉 Bronze — a bit that is always 1, and a bit that is 50/50

In [ ]:
qc = QuantumCircuit(2, 2)
qc.x(0)          # qubit 0 → always 1
qc.h(1)          # qubit 1 → 50/50
qc.measure([0, 1], [0, 1])
counts = run(qc)
print(counts)    # Qiskit reads right to left: rightmost digit = qubit 0
plot_histogram(counts)

**Result:** only `01` and `11` — the rightmost digit (qubit 0) is always 1, the left digit (qubit 1) is random.
**What it proves:** X deterministically flips 0 → 1, while H creates a superposition that measures 0 or 1 with equal probability.

### 🥈 Silver — looks uncertain, but is secretly certain

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.h(0)
qc.measure(0, 0)
print("H then H:", run(qc))           # → always 0

# Comparison: measuring in the middle breaks it
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
qc.h(0)
qc.measure(0, 0)
print("H, measure, H:", run(qc))      # → 50/50

**Result:** H·H gives `0` every time; measuring between the two H gates gives 50/50.
**What it proves:** the two H gates interfere and cancel, so a state that looks random is actually certain — unless a measurement collapses it first.

### 🥇 Gold — the vault code 101, built from 000

In [ ]:
qc = QuantumCircuit(3, 3)
qc.x([0, 2])     # flip qubits 0 and 2
qc.measure([0, 1, 2], [0, 1, 2])
print(run(qc))   # → {'101': 1024}

**Result:** `101` on every shot.
**What it proves:** X on qubits 0 and 2 deterministically produces 101; Qiskit prints qubit 0 as the rightmost digit (e.g. `qc.x(0)` alone gives `001`).

---
## Module 03 · The Two Observers *(mandatory)*
Gates allowed: H · X · CNOT · measure.

### 🥉 Bronze — a pair that only ever agrees

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
counts = run(qc)
print(counts)    # → only '00' and '11'
plot_histogram(counts)

**Result:** only `00` and `11`, never `01` or `10`.
**What it proves:** H then CNOT creates an entangled Bell pair — the two qubits always agree.

### 🥈 Silver — a pre-agreed secret also always agrees

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.measure(0, 0)     # coin flipped and looked at → now an ordinary bit
qc.cx(0, 1)          # copy that bit to the other person
qc.measure([0, 1], [0, 1])
print(run(qc))       # → only '00' and '11' too

**Result:** also only `00` and `11` — identical to the real pair.
**What it proves:** "always agreeing" alone does not prove a quantum link — it is more like a shared secret key both sides agreed on beforehand. The only difference from Bronze is measuring *before* the CNOT.

### 🥇 Gold — an experiment that tells them apart

In [ ]:
# Real quantum pair, measured in the +/- basis (H before measuring)
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.h([0, 1])
qc.measure([0, 1], [0, 1])
quantum = run(qc)
print("Quantum pair:", quantum)   # → still only '00' and '11'

# Pre-agreed secret, measured the same way
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.measure(0, 0)
qc.cx(0, 1)
qc.h([0, 1])
qc.measure([0, 1], [0, 1])
secret = run(qc)
print("Secret coin: ", secret)    # → all four outcomes

plot_histogram([quantum, secret], legend=["Quantum pair", "Secret coin"])

**Result:** the quantum pair still agrees 100%; the secret-coin pair gives all four outcomes (~50% agreement).
**What it proves:** the entangled pair agrees no matter how you measure, while a pre-agreed secret only agrees in the basis it was fixed in.

**Limitation:** a smarter cheat could pre-agree answers for *both* bases; ruling out every such cheat needs the full Bell/CHSH test, which uses rotation gates not available here.
**Link to ECHO:** each observer alone still sees 50/50 noise and cannot choose their result — correlation is real, but it carries no message.

---
## Module 05 · The Damaged Message *(mandatory)*
Each bit was sent three times; at most one of each three was flipped.

In [ ]:
stream = """100 011 001 001 001 011 100 110 001 011 000 000 001 100 011 011
000 111 000 000 110 000 001 000 000 111 001 010 101 011 111 111"""

def decode(triples):
    # Step 1: majority vote each triple → 1 bit
    OG = []
    for t in triples:
        if t.count("1") >= 2:
            OG.append("1")
        else:
            OG.append("0")
    bits = "".join(OG)
    # Step 2: every 8 bits → 1 letter
    message = ""
    for i in range(0, len(bits), 8):
        message += chr(int(bits[i:i+8], 2))
    return bits, message

### 🥉 Bronze — recover the original message

In [ ]:
bits, message = decode(stream.split())
print("Recovered bits:", bits)
print("Message:", message)

**Result:** the 32 recovered bits decode to the word **ECHO**.
**What it proves:** with at most one flip per triple, majority vote recovers every original bit.

### 🥈 Silver — where the trick breaks

In [ ]:
broken = stream.split()
print(broken[0], "→ votes 0 (correct)")
broken[0] = "110"                     # two of the three bits corrupted
print(broken[0], "→ votes 1 (WRONG)")

bits, message = decode(broken)
print("Message now:", message)        # first letter is corrupted

**Result:** changing triple 0 from `100` to `110` makes the vote pick 1, and the first letter turns into a garbage character.
**What it proves:** if 2 of 3 copies flip, the wrong value wins the vote — and it fails silently, because `110` looks like a normal triple.

### 🥇 Gold (part 1) — how many flips can it correct vs. notice?

In [ ]:
# Original bit = 0, so the sender sent "000"
tests = {
    "0 flips": "000",
    "1 flip ": "100",
    "2 flips": "110",
    "3 flips": "111",
}
for name, received in tests.items():
    vote = "1" if received.count("1") >= 2 else "0"
    noticed = received not in ["000", "111"]   # mixed = damage spotted
    correct = (vote == "0")
    print(f"{name}: {received} → vote {vote} | noticed: {noticed} | fixed right: {correct}")

**Answer:** only `000` or `111` are ever sent, so damage is noticed only when a triple is mixed.
- **Corrects 1 flip** — the two unflipped bits still win the vote.
- **Notices 2 flips** but fixes them wrongly — the flipped bits are now the majority.
- **Misses 3 flips** — `000` → `111` looks like a valid triple.

### 🥇 Gold (part 2) — why can't you protect a qubit by copying it three times?
1. **You can't copy a qubit.** Quantum rules (no-cloning) don't allow it. The closest we can get is CNOT, which links the qubits together instead of copying.
2. **You can't vote without looking.** To vote, you have to measure each copy, and measuring turns the qubit into a plain 0 or 1, so the quantum information is lost.
3. **Qubits have a hidden kind of error.** Besides flipping 0 ↔ 1, a qubit can get a sign flip (+ ↔ −). Measuring 0/1 can't see it, so voting would miss it completely.

Real quantum codes instead link the qubits together and only check "do these two match?" (parity / syndrome checks), never "what is the value?".

In [ ]:
# Proof that CNOT does not copy: try to copy "+" and "-"
qc = QuantumCircuit(2, 2)
qc.h(0)              # make +
qc.cx(0, 1)          # try to copy
qc.h([0, 1])         # ask "+ or -?"
qc.measure([0, 1], [0, 1])
print("Copy +:", run(qc), "| a real copy would give 00 only")

qc = QuantumCircuit(2, 2)
qc.x(0)
qc.h(0)              # make -
qc.cx(0, 1)          # try to copy
qc.h([0, 1])         # ask "+ or -?"
qc.measure([0, 1], [0, 1])
print("Copy -:", run(qc), "| a real copy would give 11 only")

**Result:** copying + gives `00`/`11`; copying − gives `01`/`10`.
**What it proves:** CNOT makes an entangled pair, not a copy — no-cloning holds for both + and −.

---
## Module 04 · The Erased Measurement *(stretch)*
Gates allowed: H · X · measure. Single qubit.

### 🥉 Bronze — one recoverable sequence, one destroyed

In [ ]:
# Recoverable: H then H
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.h(0)
qc.measure(0, 0)
recoverable = run(qc)
print("H, H:          ", recoverable)   # → always 0

# Destroyed: measuring in the middle
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)   # collapses the mix
qc.h(0)
qc.measure(0, 0)
destroyed = run(qc)
print("H, measure, H: ", destroyed)     # → 50/50

plot_histogram([recoverable, destroyed], legend=["H, H", "H, measure, H"])

**Result:** without a measurement in the middle the original 0 comes back every time; with one, the result is 50/50.
**What it proves:** gates are reversible, but measuring a superposition collapses it and the original state cannot be recovered.

### 🥈 Silver — predict, then run: X·H·H and H·X·H

In [ ]:
qc = QuantumCircuit(1, 1)
qc.x(0); qc.h(0); qc.h(0)
qc.measure(0, 0)
print("X, H, H:", run(qc))   # prediction: always 1

qc = QuantumCircuit(1, 1)
qc.h(0); qc.x(0); qc.h(0)
qc.measure(0, 0)
print("H, X, H:", run(qc))   # prediction: always 0

**Explanation:**
X changes the qubit from 0 to 1, then H puts it into a "−" mix because of the 1 it had. Doing H again recovers the value 1 from the "−" mix.

Running H first puts the qubit into a "+" mix based on its initial value 0. X applied to the mix doesn't change anything: X swaps 0 and 1, but "+" is an equal mix of both, so swapping leaves it the same. H is applied again and recovers the initial value 0.

The erasing is done by **measuring a mix rather than a plain bit**: it collapses the mix into a random 0 or 1, which can't be recovered. Both H and X are reversible; measurement on a mix is not.

**What it proves:** the log "changing when observed" is real physics (measurement collapse), not by itself proof of tampering.

### 🥇 Gold — prediction rules for a judge's new sequence
- H on 0 → **+**  H on 1 → **−**  H on + → **0**  H on − → **1**
- X on + → exactly unchanged  X on − → unchanged except an invisible global sign
- X·X and H·H cancel out
- Any measurement in the middle turns + or − into a random 0/1; an H afterwards makes the final result 50/50

---
## Summary
| Module | Key result | What it proves |
|---|---|---|
| 01 | X → always 1, H → 50/50, H·H → always 0, X on 0 & 2 → 101 | Gates are deterministic or interfere; randomness can hide certainty |
| 03 | Entangled pair agrees in every basis; pre-agreed secret only in one | Entanglement is real correlation, but each side alone sees only noise |
| 05 | Majority vote recovers **ECHO**; fails at 2 flips; CNOT can't copy | Classical repetition works; qubits can't be copied or voted on |
| 04 | Gates are reversible; measurement erases | "Changed when observed" is physics, not proof of tampering |

**Crux for the verdict:** correlation is not communication — entanglement alone cannot send a message (no-signaling).